# Exercise 5 — Word Embeddings and Word2Vec

One-hot has independent coordinates; Task 06 showed vocabulary mismatch. Word2Vec learns dense word representations from context. This small synthetic corpus does not produce production-quality semantic judgments. All algorithms are imported from shared source; caching/PCA are starter infrastructure.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.embeddings import DEFAULT_WORD2VEC, TRAINING_SETTINGS, prepare_tokenized_sentences, train_word2vec, get_word_vector, find_similar_words, calculate_word_similarity, project_word_vectors
from src.classical_nlp import one_hot_encode, cosine_similarity

## 1–3. Load and tokenize the transparent corpus — 5.1
Existing course names/descriptions and academic prose supply sentence-like spans; tables, headings and repeated boilerplate are removed. Inspect provenance. No metadata ground truth, expected benchmark answers or internet text. Stopwords/forms are retained; preprocessing is reused.

In [ ]:
corpus = json.loads((ROOT / "data/examples/word2vec_corpus.json").read_text(encoding="utf-8"))
raw_sentences = [item["text"] for item in corpus["sentences"]]
sentences = prepare_tokenized_sentences(raw_sentences)
print(corpus["disclaimer"])
print(len(sentences), "training spans")
list(zip(raw_sentences[:5], sentences[:5]))

## 4–5. Train CBOW — 5.2
Context → target: natural ___ processing predicts language. Gensim handles the prediction network and optimization. Defaults: 50 dimensions, window 5, min_count 1, 100 epochs, seed 42, workers 1. Stable initialization hash supports repeatability. Do not expect exact floats across platforms.

In [ ]:
DEFAULT_WORD2VEC, TRAINING_SETTINGS

In [ ]:
cbow = train_word2vec(sentences)
vocabulary = sorted(cbow.wv.index_to_key)
len(vocabulary), vocabulary[:20]

## 6. Dense vector — 5.3
One-hot dimension equals vocabulary size; the dense vector has configured dimension. Individual learned coordinates do not carry simple human-readable labels.

In [ ]:
vector = get_word_vector(cbow, "language")
print("Dense dimensions", len(vector), "norm", cosine_similarity(vector, vector, explain=True)["magnitude_a"])
print("First ten coordinates", vector[:10])
print("One-hot dimensions", len(one_hot_encode("language", vocabulary)), "active index", vocabulary.index("language"))

## 7–8. Similarity — 5.5
Reuse manual cosine from Task 06, then compare Gensim. Similarity is learned association, not necessarily synonymy or a factual relationship.

In [ ]:
pairs = [("language", "text"), ("machine", "learning"), ("machine", "database"), ("student", "course"), ("attendance", "examination")]
measurements = [{"pair": f"{a}/{b}", "manual": calculate_word_similarity(cbow, a, b), "gensim": float(cbow.wv.similarity(a, b))} for a, b in pairs]
assert all(abs(row["manual"] - row["gensim"]) < 1e-6 for row in measurements)
measurements

## 9. Nearest words — 5.4
Close in vector space does not imply interchangeable words. Observe noisy neighbours instead of hiding them.

In [ ]:
find_similar_words(cbow, "language", 5)

## 10–11. Skip-Gram and comparable architectures
Target → context: language predicts natural and processing. Change only sg from 0 to 1. Neither architecture is universally superior; separately trained coordinate axes are not aligned.

In [ ]:
skipgram = train_word2vec(sentences, sg=1)
{"CBOW": find_similar_words(cbow, "language", 5), "Skip-Gram": find_similar_words(skipgram, "language", 5)}

## 12. Window and vector-size experiments
Try smaller/larger context windows and 20/50 dimensions. Local versus topical influence is an intuition, not guaranteed behaviour. Larger vectors need more data and may be noisy. Fill the observation table before comparing conclusions with classmates.

In [ ]:
window_two = train_word2vec(sentences, window=2)
size_twenty = train_word2vec(sentences, vector_size=20)
{"Window 2": find_similar_words(window_two, "language", 5), "Window 5": find_similar_words(cbow, "language", 5), "Size 20": find_similar_words(size_twenty, "language", 5), "Size 50": find_similar_words(cbow, "language", 5)}

| Experiment | Setting A | Setting B | Observation |
|---|---|---|---|
| Architecture | CBOW | Skip-Gram | Write what changed |
| Window | 2 | 5 | Write what changed |
| Vector size | 20 | 50 | Write what changed |

## 13. Starter PCA, no student implementation
**2D projection for visualization only.** PCA fits selected words; it discards dimensions and can distort pairwise closeness. The shared helper supplies coordinates; the provided Streamlit lab renders labelled points. The optional notebook cell renders the same points using an already-installed Streamlit chart dependency, without duplicating PCA.

In [ ]:
projection_words = ["language", "text", "machine", "learning", "data", "student", "course", "examination", "attendance", "database"]
points = project_word_vectors(cbow, projection_words)
points

In [ ]:
import altair as alt
chart_data = alt.Data(values=points)
base = alt.Chart(chart_data).encode(x=alt.X("x:Q", title="PCA component 1"), y=alt.Y("y:Q", title="PCA component 2"))
chart = base.mark_point().encode(tooltip=["word:N"]) + base.mark_text(dy=-10).encode(text="word:N")
chart.to_dict()  # validates the chart; display chart in Jupyter with the next line
chart

## 14–15. GloVe and FastText, conceptual only
Word2Vec predicts using local contexts. GloVe uses global co-occurrence statistics. FastText uses character subwords; learn/learning/learner share pieces, helping rare forms, morphology and sometimes unseen forms. No pretrained vectors or extra models are downloaded.

## 16. One word → one static vector
bank approved the loan

students sat on the river bank

Traditional Word2Vec stores one bank vector if bank is known, regardless of sentence. Our workshop corpus need not contain bank: the structural limitation still holds. Verify below with the known word language in two source sentences. The retrieval API accepts a word, not its sentence context.

In [ ]:
contexts = [text for text, tokens in zip(raw_sentences, sentences) if "language" in tokens][:2]
retrieved = [get_word_vector(cbow, "language") for _ in contexts]
assert retrieved[0] == retrieved[1]
contexts, "Same stored language vector in both contexts"

## OOV behaviour
Unlike FastText, ordinary Word2Vec cannot construct an unseen form. Retrieval and similarity functions raise a readable ValueError; case is significant because training normalizes to lowercase.

In [ ]:
try:
    get_word_vector(cbow, "zzworkshopunknown")
except ValueError as error:
    print(error)

## 17. Checkpoint and next question
Checkpoint 4 (word-representation milestone): Words have learned vector representations. The original end-to-end semantic-search checkpoint remains pending.

Run `python -m pytest tests/test_embeddings.py`; suggested commit `git commit -m "Explore Word2Vec embeddings"`.

Word2Vec gives us vectors for individual words, but retrieval needs vectors for questions, sentences and documents. Next we will represent entire sentences directly. No averaging, sentence embeddings or semantic search is implemented here.